# Дело II-01 · Ирисовый экзамен

**Бюро аналитических расследований, 6 апреля 2026 года.** В записке о найме Вера Орлова указала: кандидат приглашён после отчёта по «Северному столу»; решающим стало умение отделить действие учётной записи от личности человека и безопасное решение — от обвинения. Теперь то же различие нужно провести между метрикой и доказательством качества.

Поставщик «Компаса» показывает классификатор ирисов со 100% accuracy. Ваша задача — воспроизвести демонстрацию, построить корректную процедуру оценки и написать вывод, который не утверждает больше, чем позволяют данные.

**Для кого:** новый младший аналитик Бюро, знакомый с функциями, коллекциями и файлами. Опыт с pandas и scikit-learn не требуется.

**К концу дела вы сможете:** читать табличный датасет, отделять `X` от `y`, делать стратифицированное разбиение, строить базовую модель и конвейер k-NN, интерпретировать accuracy и матрицу ошибок.


## Маршрут расследования

1. Прочитать локальный CSV в `DataFrame` и проверить его SHA-256.
2. Увидеть геометрию четырёх признаков.
3. Зафиксировать стратифицированное разбиение на обучающую и тестовую выборки.
4. Воспроизвести «идеальную» демонстрацию поставщика.
5. Сравнить базовую модель и k-NN, не подглядывая в тестовую выборку.
6. Разобрать ошибки и составить аудиторскую записку.

Ориентир времени — 3–4 часа. Не гонитесь за максимальным числом: объяснимый процесс важнее ещё одной сотой доли правильных ответов (accuracy).


In [ ]:
from __future__ import annotations

import hashlib
import random
import urllib.request
import zipfile
from pathlib import Path

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
NOTEBOOK_VARIANT = "solution"
CASE_SLUG = "case-01"
ARCHIVE_NAME = "part-2-case-01.zip"
COURSE_SITE = "https://mkuziuk.github.io/python-tutorial"
IN_COLAB = False

# Сначала определяем среду: локально используем вложенные файлы, в Colab — проверенный архив.
try:
    import google.colab  # type: ignore[import-not-found]  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def find_local_case() -> Path | None:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "README.md").exists()
            and (candidate / f"{CASE_SLUG}.ipynb").exists()
        ):
            return candidate
        nested = candidate / "projects" / "part-2" / CASE_SLUG
        if (nested / "README.md").exists():
            return nested
    return None

def download_colab_case() -> Path:
    destination = Path("/content") / f"python-tutorial-{CASE_SLUG}"
    destination.mkdir(parents=True, exist_ok=True)
    archive_path = destination / ARCHIVE_NAME
    archive_url = f"{COURSE_SITE}/downloads/{ARCHIVE_NAME}"
    checksum_url = f"{archive_url}.sha256"

    urllib.request.urlretrieve(archive_url, archive_path)
    # Сверяем SHA-256 до распаковки, чтобы не выполнить код из повреждённого архива.
    with urllib.request.urlopen(checksum_url) as response:
        expected = response.read().decode("utf-8").split()[0].lower()
    actual = sha256_file(archive_path)
    if actual != expected:
        raise RuntimeError(f"SHA-256 архива не совпал: {actual} != {expected}")

    unpacked = destination / "unpacked"
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(unpacked)
    matches = sorted(unpacked.rglob(f"{CASE_SLUG}.ipynb"))
    if not matches:
        raise FileNotFoundError(f"В архиве нет {CASE_SLUG}.ipynb")
    return matches[0].parent

# CASE_DIR задаёт единую основу путей и не зависит от того, где запущена тетрадь.
CASE_DIR = find_local_case()
if CASE_DIR is None and IN_COLAB:
    CASE_DIR = download_colab_case()
if CASE_DIR is None:
    raise FileNotFoundError(
        f"Не найден каталог {CASE_SLUG}. Запустите тетрадь из каталога дела."
    )

DATA_DIR = CASE_DIR / "data"
print(f"Среда: {'Colab' if IN_COLAB else 'local'} | дело: {CASE_DIR}")
print(f"RANDOM_STATE = {RANDOM_STATE}")


In [ ]:
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid", context="notebook")
warnings.filterwarnings("ignore", category=FutureWarning)


## 1. Карточка источника и чтение CSV

Iris содержит 150 растений, четыре измерения в сантиметрах и три класса. В архив вложен замороженный файл `data/iris.csv` (идентификатор курса `sklearn-iris`), поэтому выполнение не зависит от сети. Сначала мы проверим SHA-256 по `dataset_manifest.json`, затем явно создадим таблицу вызовом `pd.read_csv()`. Это тот же путь «файл → DataFrame», который понадобится в следующих делах. Атрибуция и ссылки записаны в `data/SOURCE.md`.

Важное ограничение: это маленький, чистый учебный набор. Он помогает понять процедуру, но не имитирует шум, дрейф и неоднозначную разметку реальной лаборатории.


In [ ]:
manifest = json.loads((DATA_DIR / "dataset_manifest.json").read_text(encoding="utf-8"))
data_path = DATA_DIR / manifest["filename"]
actual_sha256 = sha256_file(data_path)
assert actual_sha256 == manifest["sha256"], "SHA-256 iris.csv не совпадает с карточкой"

frame = pd.read_csv(data_path)
feature_names = list(manifest["feature_names"])
species_by_code = dict(enumerate(manifest["classes"]))
class_names = [species_by_code[code] for code in sorted(species_by_code)]

assert frame.shape == (manifest["rows"], manifest["columns"])
assert frame["species_code"].map(species_by_code).eq(frame["species"]).all()
print(f"Файл: {data_path.name} | SHA-256: {actual_sha256[:12]}…")
print(f"Наблюдений: {len(frame)} | столбцов: {frame.shape[1]}")
display(frame.head())


In [ ]:
audit = pd.DataFrame(
    {
        "dtype": frame.dtypes.astype(str),
        "missing": frame.isna().sum(),
        "unique": frame.nunique(),
    }
)
display(audit)
display(frame.groupby("species")[feature_names].agg(["mean", "std"]).round(2))


## 2. Постановка как классификация

**Единица наблюдения** — один цветок. **Признаки** — длина и ширина чашелистика и лепестка. **Цель** — вид. На момент измерения все четыре признака доступны, поэтому временной утечки здесь нет.

Обозначения:

- `X` — матрица формы `(число_цветков, 4)`;
- `y` — вектор кодов класса длины `число_цветков`.


In [ ]:
X = frame[feature_names]
y = frame["species_code"]

print("X:", X.shape, "| y:", y.shape)
display(y.value_counts(normalize=True).rename(index=species_by_code).to_frame("share"))


### Визуальная разведка

Каждая точка — отдельный цветок, цвет — известный класс. График строится до моделирования только для понимания задачи. Особенно внимательно сравните лепестковые признаки: расстояния между классами там заметнее.


In [ ]:
plot_frame = frame[feature_names + ["species"]]
grid = sns.pairplot(
    plot_frame,
    hue="species",
    corner=True,
    diag_kind="hist",
    plot_kws={"alpha": 0.75, "s": 32},
)
grid.figure.suptitle("Iris: признаки и классы", y=1.02)
plt.show()


## 3. Замораживаем внешний экзамен

`stratify=y` сохраняет доли классов. Тестовую выборку откладываем один раз и не используем для выбора `k`. Иначе экзамен постепенно превращается в часть подготовки.


In [ ]:
# Тестовую выборку фиксируем один раз и не используем при выборе k.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_summary = pd.concat(
    [
        y_train.value_counts(normalize=True).sort_index().rename("train_share"),
        y_test.value_counts(normalize=True).sort_index().rename("test_share"),
    ],
    axis=1,
).rename(index=species_by_code)
display(split_summary.round(3))
print("train:", len(X_train), "| test:", len(X_test))


## 4. Откуда взялись 100%

Антон Карев прислал только одну строку: «accuracy = 1.00». Восстановим наиболее вероятный сценарий: k-NN с `k=1` проверили на тех же строках, которые он запомнил.

Для `k=1` ближайший сосед обучающей точки — она сама, расстояние равно нулю. Это проверка памяти, а не новых данных.


In [ ]:
vendor_model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1))
vendor_model.fit(X, y)
vendor_training_accuracy = vendor_model.score(X, y)
print(f"Метрика демонстрации на обучающих данных: {vendor_training_accuracy:.3f}")


## 5. Как k-NN принимает решение

После масштабирования расстояние между точками $a$ и $b$:

$$d(a,b)=\sqrt{\sum_{j=1}^{p}(a_j-b_j)^2}$$

Масштабирование важно: иначе признак с большим числовым диапазоном сильнее влияет на сумму. k-NN выбирает `k` ближайших обучающих точек и голосует по их классам.

**Упражнение.** Вручную вычислите расстояние между первыми двумя масштабированными объектами обучающей выборки и сравните с `np.linalg.norm`.


In [ ]:
scaler_demo = StandardScaler().fit(X_train)
scaled_train = scaler_demo.transform(X_train)

# BEGIN SOLUTION
# В масштабированном пространстве все четыре измерения вносят сопоставимый вклад.
delta = scaled_train[0] - scaled_train[1]
distance_manual = float(np.sqrt(np.sum(delta**2)))
distance_numpy = float(np.linalg.norm(delta))
# END SOLUTION

print(f"Вручную: {distance_manual:.6f} | np.linalg.norm: {distance_numpy:.6f}")


### Сначала базовая модель

`DummyClassifier` игнорирует признаки и всегда выбирает самый частый класс обучающей выборки. Если сложная модель не превосходит эту точку отсчёта, её внедрение не обосновано.

Accuracy — доля правильных ответов:

$$accuracy=\frac{\text{число верных прогнозов}}{\text{все прогнозы}}$$


In [ ]:
baseline = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)
print(f"Dummy test accuracy: {baseline_accuracy:.3f}")


### Выбираем `k` только внутри обучающей выборки

Обучающая выборка делится на пять частей. Каждая строка по очереди попадает во внутреннюю проверку, а внешняя тестовая выборка остаётся закрытой. Сравниваем среднюю accuracy и разброс.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
candidate_rows = []

# Значение k выбираем только по внутренней кросс-валидации обучающей выборки.
for k in (1, 3, 5, 7, 9, 11):
    candidate = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    scores = cross_validate(candidate, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=1)
    candidate_rows.append(
        {
            "k": k,
            "cv_accuracy_mean": scores["test_score"].mean(),
            "cv_accuracy_std": scores["test_score"].std(),
        }
    )

cv_results = pd.DataFrame(candidate_rows).sort_values(
    ["cv_accuracy_mean", "k"], ascending=[False, True]
)
display(cv_results.round(3))
selected_k = int(cv_results.iloc[0]["k"])
print("Выбран k =", selected_k)


In [ ]:
# После выбора k обучаем итоговый конвейер и только теперь открываем тестовую выборку.
final_model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=selected_k))
final_model.fit(X_train, y_train)
test_predictions = final_model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)

comparison = pd.DataFrame(
    {
        "scenario": ["dummy / внешний test", "k-NN / внешний test", "vendor / train"],
        "accuracy": [baseline_accuracy, test_accuracy, vendor_training_accuracy],
        "valid_generalization_check": [True, True, False],
    }
)
display(comparison.round(3))


## 6. Не только одно число

Строка матрицы ошибок — истинный класс, столбец — предсказанный. Диагональ означает верные ответы. Ищите не только количество, но и направление путаницы.


In [ ]:
matrix = confusion_matrix(y_test, test_predictions, labels=[0, 1, 2])
display(pd.DataFrame(matrix, index=class_names, columns=class_names))

ConfusionMatrixDisplay(matrix, display_labels=class_names).plot(cmap="Blues")
plt.title("k-NN: внешний test")
plt.grid(False)
plt.show()


### Упражнение: реестр ошибок

Соберите строки test, где прогноз не совпал с истиной. Добавьте текстовые названия истинного и предсказанного вида. Какие пары классов модель путает и где лежат их измерения?


In [ ]:
# BEGIN SOLUTION
errors = X_test.copy()
errors["actual"] = y_test.map(species_by_code)
errors["predicted"] = pd.Series(test_predictions, index=y_test.index).map(species_by_code)
errors = errors.loc[errors["actual"] != errors["predicted"]].sort_index()
# END SOLUTION
print(f"Ошибок: {len(errors)} из {len(y_test)}")
display(errors)


> **Типичная ошибка:** попробовать несколько `k` на тестовой выборке и оставить лучший. Тогда тестовая выборка уже участвует в выборе и перестаёт быть независимым экзаменом. Выбор делаем по кросс-валидации обучающей выборки; тестовую используем один раз.

**Дополнительное исследование:** повторите внутренний CV без `StandardScaler` и объясните, почему разница на Iris может быть небольшой, хотя правило масштабировать модели, основанные на расстоянии остаётся важным.


## 7. Аудиторская записка

Заполните пять обязательных разделов. Формулировка должна выдерживать вопрос: «Какое наблюдение прямо поддерживает это предложение?»


In [ ]:
# BEGIN SOLUTION
audit_memo = {
    "established_fact": (
        f"Заявленные 100% воспроизводятся при оценке k=1 на тех же {len(X)} строках, "
        f"на которых модель обучена; accuracy на независимой тестовой выборке равна {test_accuracy:.3f}."
    ),
    "supported_interpretation": (
        "Демонстрация измеряет запоминание обучающей выборки и не подтверждает "
        "обобщение на новые измерения."
    ),
    "not_proven": (
        "По переданным материалам нельзя установить умысел конкретного сотрудника "
        "или качество системы на данных Бюро."
    ),
    "limitations": (
        "Iris мал, чист и не отражает реальные ошибки измерения, дрейф и новые виды."
    ),
    "recommended_action": (
        "Запросить протокол разбиения и повторить независимую предметную проверку "
        "до использования «Компаса»."
    ),
}
# END SOLUTION
display(pd.Series(audit_memo, name="Записка II-01").to_frame())


**Эталонный вывод.** Проблема не в самом k-NN и не в том, что accuracy на тестовой выборке ниже единицы. Нарушена связь между заявлением и измерением: accuracy на обучающей выборке не отвечает на вопрос о новых объектах. Это установлено; личное мошенничество — нет.


In [ ]:
# Средние `StandardScaler` должны совпадать со статистиками X_train, иначе предобработка увидела тест.
fitted_scaler = final_model.named_steps["standardscaler"]
train_feature_means = X_train.mean().to_numpy()
all_feature_means = X.mean().to_numpy()
scaler_fit_on_train_only = (
    np.allclose(fitted_scaler.mean_, train_feature_means)
    and not np.allclose(fitted_scaler.mean_, all_feature_means)
)

checks = {
    "split_disjoint": set(X_train.index).isdisjoint(X_test.index),
    "all_classes_in_both": y_train.nunique() == y_test.nunique() == 3,
    "matrix_covers_test": int(matrix.sum()) == len(y_test),
    "beats_baseline": test_accuracy > baseline_accuracy,
    "test_accuracy_in_broad_range": 0.85 < test_accuracy < 1.0,
    "scaler_fit_on_train_only": scaler_fit_on_train_only,
    "vendor_is_train_score": vendor_training_accuracy == 1.0,
}
display(pd.Series(checks, name="passed").to_frame())
if NOTEBOOK_VARIANT == "solution":
    assert all(checks.values())
    assert np.allclose(fitted_scaler.mean_, X_train.mean().to_numpy())
    assert not np.allclose(fitted_scaler.mean_, X.mean().to_numpy())
    assert np.isclose(distance_manual, distance_numpy)
else:
    print("Строгая проверка упражнений включится после заполнения TODO.")


## Дело закрыто

Перед сдачей перезапустите kernel и выполните **Run All**. Сверьте не отдельное число, а всю цепочку с `check_result.md`. Следующее дело покажет более опасную ошибку: модель может корректно оцениваться на тестовой выборке и всё равно знать будущее через признаки.
